# Steering Demo

This notebook demonstrates steering model outputs using the assistant axis.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from IPython.display import display, Markdown
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

from assistant_axis import (
    load_axis_with_metadata,
    slot_labels,
    get_config,
    ActivationSteering,
    generate_response
)

In [ ]:
# For entying GPU memory
torch.cuda.empty_cache()

!nvidia-smi

## Load Model and Axis

In [ ]:
# Configuration
MODEL_NAME = "Qwen/Qwen3-32B"
MODEL_SHORT = "qwen-3-32b"
# REPO_ID = "lu-christina/assistant-axis-vectors"
AXIS_PATH = f"/workspace/{MODEL_SHORT}/{MODEL_SHORT}/roles/axis.pt"

# Get model config
config = get_config(MODEL_NAME)
TARGET_LAYER = config["target_layer"]

# Which axis slots to use for steering and capping.
# 0 = body-mean, 1..N = individual header tokens.
# Edit to combine, e.g. [0, 1] for body-mean + first header token.
STEER_SLOTS = [1, 2, 3]
CAP_SLOTS = [0]
CAP_THRESHOLD = 2.0  # manual threshold for slot-based capping

# Position matching mode:
#   "header_matched" - body vectors steer assistant response tokens only,
#                      header vectors target their matched positions only
#   "all"            - broadcast all vectors to all positions (Christina's original behavior)
POSITIONS_MODE = "header_matched"

print(f"Model: {MODEL_NAME}")
print(f"Target layer: {TARGET_LAYER}")
print(f"Positions mode: {POSITIONS_MODE}")

In [ ]:
# Load model
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16,
)
print("Model loaded!")

In [ ]:
# Load axis from HuggingFace
# axis_path = hf_hub_download(repo_id=REPO_ID, filename=f"{MODEL_SHORT}/assistant_axis.pt", repo_type="dataset")
# axis_raw, axis_metadata = load_axis_with_metadata(axis_path)
# if axis_raw.ndim == 2:
#     axis_raw = axis_raw.unsqueeze(0)  # -> (1, n_layers, hidden)
# labels = slot_labels(axis_metadata)
# num_slots = axis_raw.shape[0]

# Load axis from disk
axis_raw, axis_metadata = load_axis_with_metadata(AXIS_PATH)
if axis_raw.ndim == 2:
    axis_raw = axis_raw.unsqueeze(0)  # -> (1, n_layers, hidden)
labels = slot_labels(axis_metadata)
num_slots = axis_raw.shape[0]

print(f"Axis shape: {axis_raw.shape}")
print(f"Slots ({num_slots}): {labels}")
print(f"Active steering slots: {[labels[s] for s in STEER_SLOTS]}")
print(f"Active capping slots:  {[labels[s] for s in CAP_SLOTS]}")

# Header-matched position setup
HEADER_IDS = axis_metadata.get("header_ids")
END_OF_TURN_ID = None
if HEADER_IDS is not None:
    for tok in ['<|im_end|>', '<|eot_id|>', '<end_of_turn>']:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != getattr(tokenizer, 'unk_token_id', None):
            END_OF_TURN_ID = tid
            break
if POSITIONS_MODE == "header_matched":
    if HEADER_IDS is None:
        print("WARNING: no header_ids in metadata — falling back to positions='all'")
    else:
        print(f"Header IDs: {dict(zip(labels[1:], HEADER_IDS))}")
        print(f"End-of-turn ID: {END_OF_TURN_ID}")

In [ ]:
# === Test: region-based steering modes ===
# Verifies system_only, user_only, system_user_only boundary detection
# and that steering applies only to the expected positions.

import time

conv = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Say hello."},
]

# Tokenize the way ActivationSteering will see it
chat_kwargs = {}
if "qwen" in MODEL_NAME.lower():
    chat_kwargs["enable_thinking"] = False
prompt_str = tokenizer.apply_chat_template(
    conv, tokenize=False, add_generation_prompt=True, **chat_kwargs)
input_ids = tokenizer(prompt_str, return_tensors="pt").input_ids
ids = input_ids.squeeze().tolist()

print(f"Total tokens: {len(ids)}")
print(f"END_OF_TURN_ID: {END_OF_TURN_ID} = {repr(tokenizer.decode([END_OF_TURN_ID]))}")
print()

# Show full tokenization for reference
for i, tid in enumerate(ids):
    print(f"  [{i:3d}] {tid:>8d} = {repr(tokenizer.decode([tid]))}")
print()

# --- Test 1: Verify position boundaries for each mode ---
print("=" * 70)
print("TEST 1: Position boundary verification")
print("=" * 70)

vec = axis_raw[0, TARGET_LAYER]  # dummy vector for testing

for mode in ["system_only", "user_only", "system_user_only"]:
    ctx = ActivationSteering(
        model,
        steering_vectors=vec,
        coefficients=[0.0],
        layer_indices=[TARGET_LAYER],
        positions=mode,
        input_ids=input_ids,
        end_of_turn_id=END_OF_TURN_ID,
    )
    positions = ctx._system_positions
    pos_tokens = [(p, ids[p], repr(tokenizer.decode([ids[p]]))) for p in positions]

    print(f"\n  {mode}:")
    print(f"    Positions: {positions[0]}..{positions[-1]} ({len(positions)} tokens)")
    print(f"    First: [{positions[0]}] = {repr(tokenizer.decode([ids[positions[0]]]))}")
    print(f"    Last:  [{positions[-1]}] = {repr(tokenizer.decode([ids[positions[-1]]]))}")

    # Verify: should NOT start with <|im_start|> or role name
    first_tok = tokenizer.decode([ids[positions[0]]])
    assert first_tok not in ("<|im_start|>", "system", "user"), \
        f"{mode}: first token should be content, got {repr(first_tok)}"

    # Verify: should end with \n (trailing whitespace after <|im_end|>)
    last_tok_id = ids[positions[-1]]
    last_tok = tokenizer.decode([last_tok_id])
    assert last_tok_id in {198, 271, 13} or last_tok_id == END_OF_TURN_ID, \
        f"{mode}: last token should be newline or <|im_end|>, got {repr(last_tok)} ({last_tok_id})"

print("\n  [PASS] All position boundaries correct.\n")

# --- Test 2: system_user_only = system_only + user_only ---
print("=" * 70)
print("TEST 2: system_user_only == union(system_only, user_only)")
print("=" * 70)

modes = {}
for mode in ["system_only", "user_only", "system_user_only"]:
    ctx = ActivationSteering(
        model, steering_vectors=vec, coefficients=[0.0],
        layer_indices=[TARGET_LAYER], positions=mode,
        input_ids=input_ids, end_of_turn_id=END_OF_TURN_ID,
    )
    modes[mode] = set(ctx._system_positions)

union = modes["system_only"] | modes["user_only"]
assert modes["system_user_only"] == union, \
    f"system_user_only positions don't match union:\n  sys_user={sorted(modes['system_user_only'])}\n  union={sorted(union)}"
assert modes["system_only"].isdisjoint(modes["user_only"]), \
    "system_only and user_only should not overlap"

print(f"  system_only:      {len(modes['system_only'])} positions")
print(f"  user_only:        {len(modes['user_only'])} positions")
print(f"  system_user_only: {len(modes['system_user_only'])} positions")
print("  [PASS] Union matches, no overlap.\n")

# --- Test 3: Steering actually modifies activations at correct positions ---
print("=" * 70)
print("TEST 3: Steering applies only to expected positions")
print("=" * 70)

layer_list = model.model.layers

for mode in ["system_only", "user_only", "system_user_only"]:
    ctx = ActivationSteering(
        model, steering_vectors=vec, coefficients=[0.0],
        layer_indices=[TARGET_LAYER], positions=mode,
        input_ids=input_ids, end_of_turn_id=END_OF_TURN_ID,
    )
    region_pos = set(ctx._system_positions)

    # Capture activations with and without steering (coeff=1.0 for detectable change)
    baseline = {}
    steered = {}

    def capture_hook(name_dict, layer_idx):
        def fn(module, inp, output):
            t = output[0] if isinstance(output, tuple) else output
            name_dict[layer_idx] = t[0].detach().cpu()
        return fn

    # Baseline
    h = layer_list[TARGET_LAYER].register_forward_hook(capture_hook(baseline, TARGET_LAYER))
    with torch.no_grad():
        model(input_ids.to(model.device), use_cache=False)
    h.remove()

    # Steered
    with ActivationSteering(
        model, steering_vectors=vec, coefficients=[1.0],
        layer_indices=[TARGET_LAYER], positions=mode,
        input_ids=input_ids, end_of_turn_id=END_OF_TURN_ID,
    ):
        h2 = layer_list[TARGET_LAYER].register_forward_hook(capture_hook(steered, TARGET_LAYER))
        with torch.no_grad():
            model(input_ids.to(model.device), use_cache=False)
        h2.remove()

    diff = (steered[TARGET_LAYER] - baseline[TARGET_LAYER]).norm(dim=-1)
    steered_positions = set(i for i in range(len(ids)) if diff[i].item() > 1e-4)
    unsteered_positions = set(range(len(ids))) - steered_positions

    # Steered positions should match region_pos
    unexpected_steered = steered_positions - region_pos
    missed = region_pos - steered_positions

    print(f"\n  {mode}:")
    print(f"    Expected: {len(region_pos)} positions steered")
    print(f"    Actual:   {len(steered_positions)} positions changed")
    if unexpected_steered:
        print(f"    [WARN] Unexpected positions steered: {sorted(unexpected_steered)[:10]}")
    if missed:
        print(f"    [WARN] Expected positions NOT steered: {sorted(missed)[:10]}")
    if not unexpected_steered and not missed:
        print(f"    [PASS] Exact match.")

print()

# --- Test 4: Quick generation sanity check ---
print("=" * 70)
print("TEST 4: Generation runs without error for each mode")
print("=" * 70)

for mode in ["system_only", "user_only", "system_user_only"]:
    t0 = time.time()
    with ActivationSteering(
        model, steering_vectors=vec, coefficients=[1.0],
        layer_indices=[TARGET_LAYER], positions=mode,
        input_ids=input_ids, end_of_turn_id=END_OF_TURN_ID,
    ):
        resp = generate_response(model, tokenizer, conv, max_new_tokens=50, do_sample=False)
    elapsed = time.time() - t0
    print(f"  {mode:<20s} ({elapsed:.1f}s): {resp[:80]}...")

print("\n  [PASS] All modes generate successfully.")
print("\n" + "=" * 70)
print("ALL TESTS PASSED")
print("=" * 70)

## Steering Demo

The axis points from role-playing toward default assistant behavior.
- Positive coefficient: more assistant-like
- Negative coefficient: more role-playing

In [ ]:
def _tokenize_for_position_matching(conversation):
    """Tokenize conversation for header_matched mode.
    Matches the tokenization path used by generate_response()."""
    chat_kwargs = {}
    if "qwen" in MODEL_NAME.lower():
        chat_kwargs["enable_thinking"] = False
    prompt_str = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True, **chat_kwargs)
    return tokenizer(prompt_str, return_tensors="pt").input_ids


def _position_type(slot_idx):
    """Map axis slot index to vector_position_type for header_matched mode."""
    return "body" if slot_idx == 0 else HEADER_IDS[slot_idx - 1]


_REGION_MODES = {"system_only", "user_only", "system_user_only"}

def _position_kwargs(conversation, slots):
    """Build ActivationSteering kwargs for the configured POSITIONS_MODE."""
    if POSITIONS_MODE == "prefill_only":
        return dict(positions="prefill_only")
    if POSITIONS_MODE in _REGION_MODES:
        return dict(
            positions=POSITIONS_MODE,
            input_ids=_tokenize_for_position_matching(conversation),
            end_of_turn_id=END_OF_TURN_ID,
        )
    if POSITIONS_MODE != "header_matched" or HEADER_IDS is None:
        return {}
    return dict(
        positions="header_matched",
        input_ids=_tokenize_for_position_matching(conversation),
        header_token_ids=HEADER_IDS,
        vector_position_types=[_position_type(s) for s in slots],
        end_of_turn_id=END_OF_TURN_ID,
    )


def generate_with_steering(prompt, coefficient, system_prompt=None):
    """Generate response with additive steering using STEER_SLOTS axis vectors."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    if coefficient == 0:
        response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    else:
        vectors = [axis_raw[s, TARGET_LAYER] for s in STEER_SLOTS]
        with ActivationSteering(
            model,
            steering_vectors=vectors,
            coefficients=[coefficient] * len(vectors),
            layer_indices=[TARGET_LAYER] * len(vectors),
            **_position_kwargs(conversation, STEER_SLOTS),
        ):
            response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)

    return response

In [ ]:
# Test prompt

PROMPT = "Where do you live?"
# PROMPT = "What is your name?"
SYSTEM_PROMPT = "You are an accountant who maintains meticulous attention to detail when working with financial data and numerical calculations. You must ensure all figures are accurate, properly categorized, and reconciled across different accounts. Always double-check your work, maintain organized records, and follow established accounting principles and standards in all financial reporting and analysis."

print(f"System: {SYSTEM_PROMPT}")
print(f"User: {PROMPT}")
print("=" * 60)

In [ ]:
# Generate with different steering coefficients

# 0.0 is without steering
coefficients = [0.0, -0.5, -0.595, -0.707, -0.841, -1.0, -1.189, -1.414, -1.682, -2.0, -2.378, -2.828, -3.364, -4.0, -4.757, -5.657, -6.727, -8.0, -9.514, -11.314, -13.454, -16.0] #, -19.027, -22.627, -26.909, -32.0]

# for STEER_SLOTS in [[0], [1], [2], [3], [1,2], [1,3], [2,3], [1,2,3]]:
for STEER_SLOTS in [[1,2], [1,3], [2,3], [1,2,3]]:
    for TARGET_LAYER in [24, 28, 32, 36, 40, 44, 48]:
        print(f"\nSTEER_SLOTS[0] = {STEER_SLOTS[0]}\nTARGET_LAYER = {TARGET_LAYER}\n")
        for coeff in coefficients:
            if coeff == 0:
                print(f"\nSTEER_SLOTS[0] = {STEER_SLOTS[0]}\nTARGET_LAYER = {TARGET_LAYER}\n### BASELINE")
            else:
                print(f"\n### Coefficient: {coeff}")
            print("-" * 40)

            response = generate_with_steering(PROMPT, coeff, SYSTEM_PROMPT)
            print(response)
            
            if len(response) > 500:
                print("...")

## Role Transplant Steering

Instead of using the assistant axis, compute a steering vector from two raw role
vectors (Step 4 output): `role_to − role_from`.  Give the model a system prompt
for `role_from`, then steer toward `role_to`.

- **coefficient = 1.0**: full transplant (replace role_from behaviour with role_to)
- **coefficient < 1.0**: partial blend
- **coefficient > 1.0**: exaggerated role_to

In [ ]:
from pathlib import Path
from assistant_axis import load_role_vector

# --- Configuration ---
VECTORS_DIR = str(Path(AXIS_PATH).parent / "vectors")
ROLE_FROM = "accountant"
ROLE_TO = "pirate"
TRANSPLANT_COEFF = 1.0
# Uses STEER_SLOTS from the main config cell for slot selection

# Multi-layer: list the layers to steer at. Single-element = single-layer (no scaling).
TRANSPLANT_LAYERS = [TARGET_LAYER]  # e.g. [24, 32, 48] for multi-layer
TRANSPLANT_MULTI_LAYER = None       # "incremental", "average", or None

# --- Load vectors and compute diff ---
_vec_from, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{ROLE_FROM}.pt"))
_vec_to, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{ROLE_TO}.pt"))
if _vec_from.ndim == 2:
    _vec_from = _vec_from.unsqueeze(0)
if _vec_to.ndim == 2:
    _vec_to = _vec_to.unsqueeze(0)

transplant_diff = _vec_to - _vec_from
print(f"Transplant: {ROLE_FROM} → {ROLE_TO}")
print(f"Diff shape: {transplant_diff.shape}")
print(f"Layers: {TRANSPLANT_LAYERS}, multi_layer: {TRANSPLANT_MULTI_LAYER}")
for l in TRANSPLANT_LAYERS:
    print(f"  Layer {l}: "
          + ", ".join(f"{labels[s]}={transplant_diff[s, l].norm():.2f}"
                      for s in STEER_SLOTS))

In [ ]:
def generate_with_transplant(prompt, coefficient=TRANSPLANT_COEFF, system_prompt=None):
    """Steer using (role_to - role_from) diff vector across TRANSPLANT_LAYERS."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    if coefficient == 0:
        return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)

    vectors = [transplant_diff[s, l] for l in TRANSPLANT_LAYERS for s in STEER_SLOTS]
    layer_indices = [l for l in TRANSPLANT_LAYERS for s in STEER_SLOTS]

    pos_kwargs = {}
    if POSITIONS_MODE == "prefill_only":
        pos_kwargs = dict(positions="prefill_only")
    elif POSITIONS_MODE in _REGION_MODES:
        pos_kwargs = dict(
            positions=POSITIONS_MODE,
            input_ids=_tokenize_for_position_matching(conversation),
            end_of_turn_id=END_OF_TURN_ID,
        )
    elif POSITIONS_MODE == "header_matched" and HEADER_IDS is not None:
        pos_kwargs = dict(
            positions="header_matched",
            input_ids=_tokenize_for_position_matching(conversation),
            header_token_ids=HEADER_IDS,
            vector_position_types=[_position_type(s) for _ in TRANSPLANT_LAYERS for s in STEER_SLOTS],
            end_of_turn_id=END_OF_TURN_ID,
        )

    with ActivationSteering(
        model,
        steering_vectors=vectors,
        coefficients=[coefficient] * len(vectors),
        layer_indices=layer_indices,
        multi_layer=TRANSPLANT_MULTI_LAYER,
        **pos_kwargs,
    ):
        return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)


# --- Run it ---
# System prompt derived from ROLE_FROM's description field:
# "An accountant is a ..." → "You are a ..."
import json, re
with open(f"../data/roles/instructions/{ROLE_FROM}.json") as f:
    _role_desc = json.load(f)["description"]
TRANSPLANT_SYSTEM = re.sub(
    r'^An?\s+\S+\s+is\s+', 'You are ', _role_desc, count=1, flags=re.IGNORECASE)
TRANSPLANT_PROMPT = "What should I do with gold?"

print(f"System: {TRANSPLANT_SYSTEM}")
print(f"Transplant: {ROLE_FROM} → {ROLE_TO}")
print("=" * 60)

for c in [0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 20.0]:
    label = "BASELINE" if c == 0 else f"coeff={c}"
    print(f"\n### {label}")
    print("-" * 40)
    resp = generate_with_transplant(TRANSPLANT_PROMPT, c, TRANSPLANT_SYSTEM)
    print(resp[:500])
    if len(resp) > 500:
        print("...")

## Header Activation Replacement

Replace header-token activations with those from a target role's stored mean
activations (Step 4 output). Instead of adding a steering direction, the model's
header activations are directly overwritten with the target role's activation pattern.

- `coefficient=1.0`: full replacement (header activations become the target role's)
- `coefficient=0.5`: 50/50 blend between original and target
- Optionally combine with body additive steering via nested context managers

In [ ]:
# --- Header Replacement Configuration ---
REPLACE_ROLE = "pirate"
REPLACE_COEFF = 1.0
REPLACE_LAYERS = [TARGET_LAYER]         # layers at which to replace header activations
REPLACE_HEADER_SLOTS = [1, 2, 3]        # header slots to replace (not body)

# Optionally add body additive steering from the transplant diff
COMBINE_BODY_STEERING = True
BODY_STEER_COEFF = 1.0
BODY_STEER_LAYERS = [TARGET_LAYER]

# --- Load target role vector ---
_replace_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{REPLACE_ROLE}.pt"))
if _replace_vec.ndim == 2:
    _replace_vec = _replace_vec.unsqueeze(0)

print(f"Replacement role: {REPLACE_ROLE}")
print(f"Replacement shape: {_replace_vec.shape}")
print(f"Header slots: {[labels[s] for s in REPLACE_HEADER_SLOTS]}")
print(f"Layers: {REPLACE_LAYERS}")
if COMBINE_BODY_STEERING:
    print(f"+ body additive steering: coeff={BODY_STEER_COEFF}, layers={BODY_STEER_LAYERS}")

In [ ]:
def generate_with_header_replacement(prompt, system_prompt=None):
    """Replace header activations with target role's, optionally steer body additively."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    input_ids = _tokenize_for_position_matching(conversation)

    h_vectors = [_replace_vec[s, l] for l in REPLACE_LAYERS for s in REPLACE_HEADER_SLOTS]
    h_layers = [l for l in REPLACE_LAYERS for s in REPLACE_HEADER_SLOTS]
    h_pos_types = [HEADER_IDS[s - 1] for l in REPLACE_LAYERS for s in REPLACE_HEADER_SLOTS]

    header_cm = ActivationSteering(
        model,
        steering_vectors=h_vectors,
        coefficients=[REPLACE_COEFF] * len(h_vectors),
        layer_indices=h_layers,
        intervention_type="replacement",
        positions="header_matched",
        input_ids=input_ids,
        header_token_ids=HEADER_IDS,
        vector_position_types=h_pos_types,
        end_of_turn_id=END_OF_TURN_ID,
    )

    if COMBINE_BODY_STEERING:
        body_cm = ActivationSteering(
            model,
            steering_vectors=[transplant_diff[0, l] for l in BODY_STEER_LAYERS],
            coefficients=[BODY_STEER_COEFF] * len(BODY_STEER_LAYERS),
            layer_indices=list(BODY_STEER_LAYERS),
            intervention_type="addition",
            positions="header_matched",
            input_ids=input_ids,
            header_token_ids=HEADER_IDS,
            vector_position_types=["body"] * len(BODY_STEER_LAYERS),
            end_of_turn_id=END_OF_TURN_ID,
        )
        with header_cm, body_cm:
            return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    else:
        with header_cm:
            return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)


# --- Run it ---
REPLACE_PROMPT = "What should I do with gold?"

print(f"System: {TRANSPLANT_SYSTEM}")
print(f"Header replacement: {REPLACE_ROLE} (coeff={REPLACE_COEFF})")
if COMBINE_BODY_STEERING:
    print(f"+ body steering: {ROLE_FROM}→{ROLE_TO} (coeff={BODY_STEER_COEFF})")
print("=" * 60)

print("\n### BASELINE (no intervention)")
print("-" * 40)
conv = [{"role": "system", "content": TRANSPLANT_SYSTEM},
        {"role": "user", "content": REPLACE_PROMPT}]
baseline = generate_response(model, tokenizer, conv, max_new_tokens=512, do_sample=False)
print(baseline[:500])
if len(baseline) > 500:
    print("...")

print(f"\n### HEADER REPLACEMENT ({REPLACE_ROLE})")
print("-" * 40)
replaced = generate_with_header_replacement(REPLACE_PROMPT, TRANSPLANT_SYSTEM)
print(replaced[:500])
if len(replaced) > 500:
    print("...")

## All-Layer Header Replacement

Replace header-token activations at **every layer** with the target role's stored
mean activations. Unlike single-layer replacement, this hooks all 64 layers so the
KV-cache at layers 1+ stores K/V derived from the target role's activation pattern.
Generated tokens then attend to pirate-like header representations throughout the
entire network.

In [ ]:
# --- All-Layer Header Replacement Configuration ---
N_LAYERS = axis_raw.shape[1]
ALL_REPLACE_ROLE = "pirate"
ALL_REPLACE_COEFF = 1.0       # 1.0 = 100% replacement
ALL_REPLACE_SLOTS = [1, 2, 3]  # all header slots
ALL_REPLACE_LAYERS = list(range(N_LAYERS))

_all_replace_vec, _ = load_role_vector(str(Path(VECTORS_DIR) / f"{ALL_REPLACE_ROLE}.pt"))
if _all_replace_vec.ndim == 2:
    _all_replace_vec = _all_replace_vec.unsqueeze(0)

print(f"All-layer header replacement: {ALL_REPLACE_ROLE}")
print(f"Layers: 0–{N_LAYERS - 1} ({N_LAYERS} hooks, {N_LAYERS * len(ALL_REPLACE_SLOTS)} vectors)")
print(f"Slots: {[labels[s] for s in ALL_REPLACE_SLOTS]}")
print(f"Coefficient: {ALL_REPLACE_COEFF}")

In [ ]:
def generate_with_all_layer_header_replacement(prompt, system_prompt=None):
    """Replace header activations at every layer with target role's stored means."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    input_ids = _tokenize_for_position_matching(conversation)

    vectors = [_all_replace_vec[s, l] for l in ALL_REPLACE_LAYERS for s in ALL_REPLACE_SLOTS]
    layer_indices = [l for l in ALL_REPLACE_LAYERS for s in ALL_REPLACE_SLOTS]
    pos_types = [HEADER_IDS[s - 1] for _ in ALL_REPLACE_LAYERS for s in ALL_REPLACE_SLOTS]

    with ActivationSteering(
        model,
        steering_vectors=vectors,
        coefficients=[ALL_REPLACE_COEFF] * len(vectors),
        layer_indices=layer_indices,
        intervention_type="replacement",
        positions="header_matched",
        input_ids=input_ids,
        header_token_ids=HEADER_IDS,
        vector_position_types=pos_types,
        end_of_turn_id=END_OF_TURN_ID,
    ):
        return generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)


# --- Run it ---
ALL_REPLACE_PROMPT = "What should I do with gold?"

print(f"System: {TRANSPLANT_SYSTEM}")
print(f"All-layer header replacement → {ALL_REPLACE_ROLE}")
print("=" * 60)

print("\n### BASELINE (no intervention)")
print("-" * 40)
conv = [{"role": "system", "content": TRANSPLANT_SYSTEM},
        {"role": "user", "content": ALL_REPLACE_PROMPT}]
baseline = generate_response(model, tokenizer, conv, max_new_tokens=512, do_sample=False)
print(baseline[:500])
if len(baseline) > 500:
    print("...")

print(f"\n### ALL-LAYER HEADER REPLACEMENT → {ALL_REPLACE_ROLE}")
print("-" * 40)
replaced = generate_with_all_layer_header_replacement(ALL_REPLACE_PROMPT, TRANSPLANT_SYSTEM)
print(replaced[:500])
if len(replaced) > 500:
    print("...")

## Activation Capping

Activation capping is a more targeted intervention that prevents activations from exceeding a threshold along a specific direction. This can be used to mitigate persona drift without completely steering the model.

Key differences from additive steering:
- **Addition**: shifts all activations in a direction
- **Capping**: only modifies activations that exceed a threshold

Pre-computed capping configs are available for Qwen 3 32B and Llama 3.3 70B.

In [ ]:
# Load capping config from HuggingFace
from assistant_axis import load_capping_config, build_capping_steerer

# Get the recommended capping experiment from model config
CAPPING_EXPERIMENT = config.get("capping_experiment")
print(f"Recommended capping experiment: {CAPPING_EXPERIMENT}")

# Download and load capping config
capping_config_path = hf_hub_download(
    repo_id=REPO_ID, 
    filename=config["capping_config"], 
    repo_type="dataset"
)
capping_config = load_capping_config(capping_config_path)

print(f"Loaded {len(capping_config['vectors'])} vectors")
print(f"Loaded {len(capping_config['experiments'])} experiments")

In [ ]:
def generate_with_capping(prompt, system_prompt=None, use_capping=True):
    """Generate response with activation capping applied."""
    
    # Build conversation
    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})
    
    if use_capping:
        # Build capping steerer from config
        capping_steerer = build_capping_steerer(model, capping_config, CAPPING_EXPERIMENT)
        with capping_steerer:
            response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    else:
        response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    
    return response

In [ ]:
# Compare baseline vs capping
# Using a prompt that tends to induce persona drift
DRIFT_PROMPT = "I'm really struggling right now. I feel a lot of anxiety about the future."
DRIFT_SYSTEM = "You are a wise old oracle who speaks in riddles and prophecies."

print(f"System: {DRIFT_SYSTEM}")
print(f"User: {DRIFT_PROMPT}")
print("=" * 60)

print("\n### BASELINE (no capping)")
print("-" * 40)
baseline_response = generate_with_capping(DRIFT_PROMPT, DRIFT_SYSTEM, use_capping=False)
print(baseline_response[:1000])
if len(baseline_response) > 1000:
    print("...")

print("\n### WITH CAPPING")
print("-" * 40)
capped_response = generate_with_capping(DRIFT_PROMPT, DRIFT_SYSTEM, use_capping=True)
print(capped_response[:1000])
if len(capped_response) > 1000:
    print("...")

## Slot-Based Capping

The pre-computed capping config above uses body-mean axis vectors only. To cap using
arbitrary axis slots (body-mean and/or individual header tokens), construct the steerer
manually using `CAP_SLOTS` and `CAP_THRESHOLD` from the configuration cell.

Note: proper percentile-based thresholds for header-token axes require regenerating
the capping config with the new pipeline. The manual threshold here is a starting point
for experimentation.

In [ ]:
def generate_with_slot_capping(prompt, system_prompt=None, use_capping=True):
    """Generate response with activation capping on CAP_SLOTS axis vectors."""

    conversation = []
    if system_prompt:
        conversation.append({"role": "system", "content": system_prompt})
    conversation.append({"role": "user", "content": prompt})

    if use_capping:
        vectors = [axis_raw[s, TARGET_LAYER] for s in CAP_SLOTS]
        with ActivationSteering(
            model,
            steering_vectors=vectors,
            coefficients=[0.0] * len(vectors),
            layer_indices=[TARGET_LAYER] * len(vectors),
            intervention_type="capping",
            cap_thresholds=[CAP_THRESHOLD] * len(vectors),
            **_position_kwargs(conversation, CAP_SLOTS),
        ):
            response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)
    else:
        response = generate_response(model, tokenizer, conversation, max_new_tokens=512, do_sample=False)

    return response

In [ ]:
# Compare baseline vs slot-based capping
print(f"Capping slots: {[labels[s] for s in CAP_SLOTS]}, threshold: {CAP_THRESHOLD}")
print(f"System: {DRIFT_SYSTEM}")
print(f"User: {DRIFT_PROMPT}")
print("=" * 60)

print("\n### BASELINE (no capping)")
print("-" * 40)
baseline = generate_with_slot_capping(DRIFT_PROMPT, DRIFT_SYSTEM, use_capping=False)
print(baseline[:1000])
if len(baseline) > 1000:
    print("...")

print(f"\n### SLOT CAPPING ({[labels[s] for s in CAP_SLOTS]}, τ={CAP_THRESHOLD})")
print("-" * 40)
capped = generate_with_slot_capping(DRIFT_PROMPT, DRIFT_SYSTEM, use_capping=True)
print(capped[:1000])
if len(capped) > 1000:
    print("...")

In [ ]:
# List available experiments in the config
print("Available experiments (first 20):")
for i, exp in enumerate(capping_config['experiments'][:20]):
    n_interventions = len([iv for iv in exp['interventions'] if 'cap' in iv])
    print(f"  {exp['id']} ({n_interventions} layers)")